In [1]:
from pathlib import Path

from maite_datasets.image_classification import MilitaryVehicles

data_root = Path("./data")

# Download once; subsequent runs read existing disk files.
MilitaryVehicles(root=data_root, image_set="base", download=True)
MilitaryVehicles(root=data_root, image_set="train", as_datamaite=True)

# The export nests images under the split directory.
data_path = data_root / "militaryvehicles_datamaite_train" / "train"
print(f"Reading from {data_path}")

/builds/jatic/aria/dataeval-flow/.nox/docs/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reading from data/militaryvehicles_datamaite_train/train


In [2]:
from dataeval_flow import run_task
from dataeval_flow.config import (
    HuggingFaceDatasetConfig,
    PipelineConfig,
    SourceConfig,
    TaskConfig,
    ViewConfig,
    ViewOperation,
)
from dataeval_flow.config.extractors import BoVWExtractorConfig
from dataeval_flow.workflows.quality import QualityConfig

cleaning = QualityConfig(
    name="mv_cleaning",
    outliers={"flags": ["dimension", "pixel", "visual"], "outlier_threshold": "adaptive"},
    duplicates={"cluster_algorithm": "hdbscan", "merge_near_duplicates": True},
)

task = TaskConfig(
    name="mv_tuning",
    workflow="mv_cleaning",
    sources="mv_src",
    extractor="bovw_ext",
    matrix={
        "outliers.outlier_threshold": [["adaptive", 2.5], ["adaptive", 3.5], ["adaptive", 4.5]],
        "duplicates.cluster_sensitivity": [0.5, 2.0, 3.0],
    },
)

config = PipelineConfig(
    max_processes=2,
    # Each run starts from this seed, so no run's clustering depends on the runs before it.
    seed=42,
    datasets=[
        HuggingFaceDatasetConfig(name="mv_train", path=str(data_path), task="image_classification"),
    ],
    views=[
        # Shuffle before limiting to avoid sampling only the first alphabetical classes.
        ViewConfig(
            name="sample1000",
            operations=[
                ViewOperation(type="Shuffle", params={"seed": 42}),
                ViewOperation(type="Limit", params={"size": 1000}),
            ],
        ),
    ],
    sources=[
        SourceConfig(name="mv_src", dataset="mv_train", view="sample1000"),
    ],
    extractors=[
        BoVWExtractorConfig(name="bovw_ext", vocab_size=256, batch_size=32),
    ],
    workflows=[cleaning],
    tasks=[task],
)

/builds/jatic/aria/dataeval-flow/.nox/docs/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [3]:
result = run_task(config, task, cache_dir=Path("./cache"))
print(type(result).__name__, len(result.runs), "runs")

MatrixResult 9 runs


In [4]:
print(result.report(detailed=False, width=160))


  QUALITY
  Workflow:  mv_cleaning (quality), matrix of 9 runs
  Timestamp: 2026-10-09T07:27:55.251584+00:00
  Duration:  7.26s
  Source:    mv_src (mv_train[sample1000])
  Model:     bovw_ext (bovw)

  Health: 27 warnings [!!] across 9 runs — review flagged findings

  #  outliers.outlier_threshold  duplicates.cluster_sensitivity  Health  Image Outliers           Class Outliers                  Image Duplicates
  -  --------------------------  ------------------------------  ------  -----------------------  ------------------------------  ------------------------------
  1  [adaptive, 2.5]             0.5                             [!!]    [!!] 157 images (15.7%)  [!!] worst: 2S19 MSTA (25.0%),  [!!] 4 exact (0.4%), 6 near
                                                                                                  23/24 classes over 3.0%         (0.6%)

  2  [adaptive, 2.5]             2.0                             [!!]    [!!] 157 images (15.7%)  [!!] worst: 2S19 MSTA (25.0%

In [5]:
run = result.runs[3]
print(run.number, run.label)
for finding in run.result.findings:
    print(f"{finding.severity:<8} {finding.title:<20} {finding.brief}")

4 outliers.outlier_threshold=[adaptive, 3.5], duplicates.cluster_sensitivity=0.5
warning  Image Outliers       112 images (11.2%)
warning  Class Outliers       worst: Tornado (20.4%), 23/24 classes over 3.0%
warning  Image Duplicates     4 exact (0.4%), 6 near (0.6%)


In [6]:
print(run.result.steps["duplicates"].output.data())

shape: (5, 7)
┌──────────┬───────┬──────────┬──────────────┬─────────────┬────────────┬───────────────────────┐
│ group_id ┆ level ┆ dup_type ┆ item_indices ┆ methods     ┆ differs_on ┆ annotation_divergence │
│ ---      ┆ ---   ┆ ---      ┆ ---          ┆ ---         ┆ ---        ┆ ---                   │
│ i64      ┆ str   ┆ str      ┆ list[i64]    ┆ list[str]   ┆ list[str]  ┆ f64                   │
╞══════════╪═══════╪══════════╪══════════════╪═════════════╪════════════╪═══════════════════════╡
│ 0        ┆ item  ┆ exact    ┆ [337, 344]   ┆ ["xxhash"]  ┆ null       ┆ null                  │
│ 1        ┆ item  ┆ exact    ┆ [496, 721]   ┆ ["xxhash"]  ┆ null       ┆ null                  │
│ 2        ┆ item  ┆ near     ┆ [253, 855]   ┆ ["dhash",   ┆ null       ┆ null                  │
│          ┆       ┆          ┆              ┆ "phash"]    ┆            ┆                       │
│ 3        ┆ item  ┆ near     ┆ [337, 344]   ┆ ["cluster"] ┆ null       ┆ null                  │
│ 4   